In [0]:
df_zones = spark.read.option("header", "true").option("inferschema", "true").csv("/Volumes/nyc/default/nyc_raw_data/taxi_zone_lookup.csv")

df_zones.write.format("delta").mode("overwrite").option("overwtiteschema", "true").saveAsTable("nyc.default.taxi_zone_lookup")

print(f"Zone Lookup Rows: {df_zones.count()}")

display(df_zones)

In [0]:
import pyspark.sql.functions as F

df_silver_final = spark.read.table("nyc.default.silver_yellow_taxi")

df_zone = spark.read.table("nyc.default.taxi_zone_lookup")

df_joined = (df_silver_final.join(df_zone.select(F.col("LocationID").alias("pu_location_id"), F.col("Zone").alias("pickup_zone"), F.col("Borough").alias("pickup_borough")), on="pu_location_id", how="left"))

print(f"Joined row count: {df_joined.count():,}")

In [0]:
#Gold table 1: trips by pickup zone
from pyspark.sql import functions as F

df_gold_zone = (
    df_joined
    .groupBy("pu_location_id", "pickup_zone", "pickup_borough")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance_miles"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue")
    )
    .orderBy(F.desc("total_trips"))
)

(
    df_gold_zone
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("nyc.default.gold_trips_by_zone")
)

print("gold_trips_by_zone written.")
display(df_gold_zone.limit(10))

In [0]:
#Gold table 2: trips by hour of day
df_gold_hour = (
    df_silver_final
    .withColumn("hour_of_day", F.hour("pickup_datetime"))
    .groupBy("hour_of_day")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance_miles"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes")
    )
    .orderBy("hour_of_day")
)

(
    df_gold_hour
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("nyc.default.gold_trips_by_hour")
)

print("gold_trips_by_hour written.")
display(df_gold_hour)

In [0]:
# Verify both Gold tables
print("gold_trips_by_zone:")
print(spark.read.table("nyc.default.gold_trips_by_zone").count())

print("gold_trips_by_hour:")
print(spark.read.table("nyc.default.gold_trips_by_hour").count())